In [ ]:
# Thiết lập cho Google Colab
import os
if not os.path.exists('/content/ML-labs'):
    !git clone https://github.com/pie-12/ML-labs.git /content/ML-labs

os.chdir('/content/ML-labs/')
print("✅ Cấu hình dữ liệu thành công! Thư mục làm việc hiện tại:", os.getcwd())

# Lab 1 - Dự án Machine Learning
**Dự án End-to-End: Từ dữ liệu thô (Raw Data) đến file model.pkl**

In [ ]:
import sys
import sklearn
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import joblib

# Yêu cầu Scikit-Learn >=0.20
assert sklearn.__version__ >= "0.20"

# Thiết lập Matplotlib
%matplotlib inline

## 1. Thu thập dữ liệu
Chúng ta sẽ sử dụng bộ dữ liệu giá nhà (Housing) đã có sẵn trong không gian làm việc.

In [ ]:
HOUSING_PATH = os.path.join("handson-ml2-vn-main", "datasets", "housing")

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing = load_housing_data()
housing.head()

## 2. Khám phá và Phân tích Dữ liệu (EDA)

In [ ]:
# Thông tin chung về dữ liệu
housing.info()

Trục hoành (X) là giá trị của đặc trưng, trục tung (Y) là số lượng mẫu (tần suất).

In [ ]:
# Vẽ biểu đồ Histogram
housing.hist(bins=50, figsize=(20,15))
plt.suptitle("Phân phối của các đặc trưng", fontsize=16)
plt.show()

In [ ]:
# Biểu đồ phân tán (Scatter plot) của các tọa độ địa lý
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"]/100, label="Dân số", figsize=(10,7),
             c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
             sharex=False)
plt.xlabel("Kinh độ (Longitude)")
plt.ylabel("Vĩ độ (Latitude)")
plt.title("Bản đồ mật độ dân số và giá nhà tại California")
plt.legend()
plt.show()

In [ ]:
# Ma trận tương quan
corr_matrix = housing.corr(numeric_only=True)
print("\nMức độ tương quan với giá nhà trung vị (median_house_value):")
print(corr_matrix["median_house_value"].sort_values(ascending=False))

## 3. Chuẩn bị dữ liệu
Xử lý các giá trị bị thiếu, mã hóa dữ liệu hạng mục và chuẩn hóa tỉ lệ.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Tạo phân chia phân tầng (stratified split) dựa trên thu nhập
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6.0, np.inf],
                               labels=[1, 2, 3, 4, 5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Tách các cột số và cột phân loại
housing_num = housing.drop("ocean_proximity", axis=1)
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

# Pipeline cho dữ liệu số: Xử lý giá trị thiếu và chuẩn hóa
num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("std_scaler", StandardScaler()),
    ])

# Pipeline hoàn chỉnh: Kết hợp cả xử lý số và phân loại
full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

housing_prepared = full_pipeline.fit_transform(housing)
print("Kích thước dữ liệu sau khi chuẩn bị:", housing_prepared.shape)

## 4. Huấn luyện mô hình
Tiến hành huấn luyện các mô hình Linear Regression, Decision Tree và Random Forest.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Hồi quy tuyến tính (Linear Regression)
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
housing_predictions = lin_reg.predict(housing_prepared)
lin_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Sai số RMSE của Linear Regression:", lin_rmse)

# Cây quyết định (Decision Tree)
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)
housing_predictions = tree_reg.predict(housing_prepared)
tree_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Sai số RMSE của Decision Tree:", tree_rmse)

# Rừng ngẫu nhiên (Random Forest)
forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
forest_reg.fit(housing_prepared, housing_labels)
housing_predictions = forest_reg.predict(housing_prepared)
forest_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Sai số RMSE của Random Forest:", forest_rmse)

## 5. Tinh chỉnh mô hình (Grid Search)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]},
]

forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring="neg_mean_squared_error",
                           return_train_score=True)

grid_search.fit(housing_prepared, housing_labels)

print("Các tham số tốt nhất:", grid_search.best_params_)

# Đánh giá trên tập kiểm tra (test set)
final_model = grid_search.best_estimator_

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)

final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
print("Sai số RMSE của mô hình cuối cùng trên tập kiểm tra:", final_rmse)

## 6. Lưu mô hình (Xuất ra file model.pkl)

In [ ]:
# Tạo pipeline đầy đủ bao gồm cả bước tiền xử lý và mô hình dự đoán
full_pipeline_with_predictor = Pipeline([
        ("preparation", full_pipeline),
        ("final_model", final_model)
    ])

# Huấn luyện lại toàn bộ pipeline trên toàn bộ tập dữ liệu (tùy chọn nhưng tốt cho production)
full_pipeline_with_predictor.fit(housing, housing_labels)

# Lưu vào tệp model.pkl
joblib.dump(full_pipeline_with_predictor, "model.pkl")
print("Đã lưu mô hình thành công vào file model.pkl")

# Kiểm tra lại: Tải lại mô hình
loaded_model = joblib.load("model.pkl")
print("Đã tải thành công mô hình từ file.")